# Random Walk Generator

Generate random walk time series with configurable drift, volatility, and start value.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from synforecast.generators import RandomWalkGenerator

## Define Generator Parameters

Configure the random walk with drift (upward trend), volatility, hourly frequency, and a fixed start value.

In [ ]:
params = {
    "min_length": 100,
    "max_length": 150,
    "freq": "h",
    "drift": 0.1,
    "volatility": 1.5,
    "start_value": 100.0,
    "seed": 42,
}

generator = RandomWalkGenerator(engine="polars", **params)
df = generator.generate(n_series=3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df["unique_id"].unique().to_list():
    series = df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("Random Walk Time Series")
ax.legend()
plt.tight_layout()
plt.show()

## Inspect the Generated Data

The output is a Polars DataFrame with columns `unique_id`, `ds` (timestamp), and `y` (value).

In [ ]:
print(f"Generated {df['unique_id'].n_unique()} time series")
print(f"Total observations: {len(df)}")
df.head(10)

## Statistics by Series

Compute summary statistics for each generated series.

In [ ]:
df.group_by("unique_id").agg(
    [
        pl.col("y").count().alias("count"),
        pl.col("y").min().alias("min_value"),
        pl.col("y").max().alias("max_value"),
        pl.col("y").mean().alias("mean_value"),
        pl.col("y").std().alias("std_value"),
    ]
)